In [77]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [78]:
df = pd.read_csv("./data/Churn_Modelling.csv")

In [79]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Preprocessing


In [80]:
df = df.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)

# Encode categorical variables
gender_label_encoder = LabelEncoder()

df["Gender"] = gender_label_encoder.fit_transform(df["Gender"])

one_hot_geography_encoder = OneHotEncoder(
    drop="first"
)  # drop='first' to avoid dummy variable trap

df = pd.concat(
    [
        df.drop("Geography", axis=1),
        pd.DataFrame(
            one_hot_geography_encoder.fit_transform(df[["Geography"]]).toarray(),
            columns=one_hot_geography_encoder.get_feature_names_out(["Geography"]),
        ),
    ],
    axis=1,
)

X = df.drop("EstimatedSalary", axis=1)
y = df["EstimatedSalary"]

In [81]:
X.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0.0,1.0


In [82]:
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

## Train Test split and Scaling


In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [84]:
X_train[:10]

array([[ 0.35649971,  0.91324755, -0.6557859 ,  0.34567966, -1.21847056,
         0.80843615,  0.64920267,  0.97481699, -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, -0.3483691 ,  0.69683765,
         0.80843615,  0.64920267,  0.97481699,  1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, -0.69539349,  0.61862909,
        -0.91668767,  0.64920267, -1.02583358, -0.57946723,  1.73494238],
       [-0.94071667, -1.09499335, -1.13114808,  1.38675281,  0.95321202,
        -0.91668767,  0.64920267, -1.02583358, -0.57946723, -0.57638802],
       [-1.39733684,  0.91324755,  1.62595257,  1.38675281,  1.05744869,
        -0.91668767, -1.54035103, -1.02583358, -0.57946723, -0.57638802],
       [-0.85769482,  0.91324755,  0.19986603, -0.3483691 ,  0.7067467 ,
        -0.91668767,  0.64920267, -1.02583358, -0.57946723, -0.57638802],
       [ 0.32536652, -1.09499335,  0.10479359, -1.38944225, -1.21847056,
         0.80843615, -1.54035103, -1.02583358

## Saving the preprocessing objects


In [92]:
with open("./models/reg_gender_label_encoder.pkl", "wb") as f:
    pickle.dump(gender_label_encoder, f)

with open("./models/reg_one_hot_geography_encoder.pkl", "wb") as f:
    pickle.dump(one_hot_geography_encoder, f)

with open("./models/reg_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

## ANN Reg Implementation


In [86]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [87]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="linear"),
    ]
)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

In [88]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_13 (Dense)                │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,249 (51.75 KB)

 Trainable params: 13,249 (51.75 KB)

 Non-trainable params: 0 (0.00 B)

In [89]:
LOG_DIR = f"logs/fit/{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[
        early_stopping_callback,
        TensorBoard(log_dir=LOG_DIR),
    ],
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 9867531264.0000 - mae: 82609.4688 - val_loss: 3469949696.0000 - val_mae: 50489.5000
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3411061760.0000 - mae: 50202.5234 - val_loss: 3399271936.0000 - val_mae: 50199.7305
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3374110464.0000 - mae: 49996.9883 - val_loss: 3391303936.0000 - val_mae: 50171.3945
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3357453568.0000 - mae: 49874.5039 - val_loss: 3388621824.0000 - val_mae: 50222.3320
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3341032192.0000 - mae: 49774.3438 - val_loss: 3410951424.0000 - val_mae: 50344.4805
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3330104320.0000 - mae: 49712.2070 - val_loss: 3379160576.0000 - val_mae: 50171.3789
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3329195776.0000 - mae: 49698.4883 - val_loss: 3367490560.0000 - val_m

In [90]:
model.save("./models/reg_churn_model.h5")

## Evaluation


In [91]:
from sklearn.metrics import r2_score

test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"Test R2 Score: {r2}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3367490560.0000 - mae: 50113.0078
Test Loss: 3367490560.0, Test MAE: 50113.0078125
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Test R2 Score: -0.020064216500301768
